# Pydantic Part 14: Enums & `use_enum_values`

When building APIs or processing data, you often want to restrict a field to a specific set of allowed values (e.g., status codes, roles, colors). While you could write a custom validator to check if a string matches a list of allowed strings, the Pythonic and Pydantic way is to use the built-in `enum` module.

## 1. Basic Enum Integration

Pydantic natively understands Python Enums. If you set a field's type hint to an Enum class, Pydantic will strictly enforce that incoming data matches one of the Enum's values.

```python
from enum import Enum
from pydantic import BaseModel, ValidationError

class Role(Enum):
    ADMIN = "admin"
    EDITOR = "editor"
    VIEWER = "viewer"

class User(BaseModel):
    role: Role

# ✅ Deserializing from JSON (Using the string value)
u1 = User.model_validate_json('{"role": "admin"}')
print(u1.role) # Output: Role.ADMIN (It loads as the Python Enum object!)

# ❌ Invalid Enum Value
try:
    User.model_validate_json('{"role": "super_user"}')
except ValidationError as e:
    print(e)
    # Output: Input should be 'admin', 'editor' or 'viewer'

```

### Serialization Behavior (The Default)

* When you call `model_dump()`, Pydantic leaves the field as the Python `Enum` object.
* When you call `model_dump_json()`, Pydantic intelligently converts the `Enum` object into its underlying value (e.g., the string `"admin"`), because JSON does not understand Python objects.

## 2. Forcing Enum Values (`use_enum_values`)

Sometimes, you want `model_dump()` to return the raw string value (e.g., `"admin"`) instead of the Python Enum object (`Role.ADMIN`). This is especially common if you are taking the dictionary output of `model_dump()` and passing it to a downstream system (like an ORM or standard `json.dumps()`) that will crash if it encounters a Python Enum object.

You can configure this at the model level using `use_enum_values=True`.

```python
from pydantic import ConfigDict

class FlatUser(BaseModel):
    model_config = ConfigDict(use_enum_values=True)
    role: Role

u2 = FlatUser(role=Role.ADMIN)

# Notice how it now returns the string, not the Enum object!
print(u2.model_dump()) # {'role': 'admin'}

```

## 3. The "Unvalidated Default" Trap with Enums

If you use `use_enum_values=True` **and** you provide a default Enum value, you will run into a bug due to Pydantic's default behavior of skipping default validation.

```python
class BuggyUser(BaseModel):
    model_config = ConfigDict(use_enum_values=True)
    
    # ❌ TRAP: Pydantic skips validation here.
    role: Role = Role.VIEWER 

u3 = BuggyUser()
print(u3.model_dump()) 
# {'role': <Role.VIEWER: 'viewer'>} -> IT FAILED TO EXTRACT THE VALUE!

```

Because validation was skipped, the system never triggered the logic to extract the value from the Enum object. You can fix this in one of two ways:

**Fix A: Hardcode the Value**

```python
role: Role = Role.VIEWER.value

```

**Fix B: Force Validation (The Safer Approach)**

```python
model_config = ConfigDict(use_enum_values=True, validate_default=True)
role: Role = Role.VIEWER

```


### Interview Preparation: Enums in Pydantic

<details>
<summary><b>1. You need to restrict an API endpoint so a user can only set their `account_tier` to "basic", "pro", or "enterprise". How would you implement this cleanly in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
I would create a standard Python `Enum` class called `AccountTier` containing those three string values. I would then use that Enum class as the type hint for the `account_tier` field in my Pydantic model. <br><br>
This is superior to writing a custom validator because Pydantic handles the constraint automatically, and it will automatically generate the correct "allowed values" list in the OpenAPI/Swagger documentation, providing a much better developer experience for the API consumers.
</details>

<details>
<summary><b>2. You are using `model_dump()` to convert a Pydantic model into a dictionary, and then you pass that dictionary into standard `json.dumps()`. The application crashes with a `TypeError: Object of type Role is not JSON serializable`. Why did this happen and how do you fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This happens because standard Pydantic models keep Enum fields as native Python `Enum` objects when you call `model_dump()`. The standard Python `json.dumps()` module does not know how to serialize Python Enum objects. <br><br>
There are two ways to fix this. The best way is to stop using `json.dumps()` entirely and just use Pydantic's built-in `model.model_dump_json()`, which handles Enum extraction natively. If you must use `json.dumps()` downstream, you can fix it by adding `model_config = ConfigDict(use_enum_values=True)` to your Pydantic model. This forces `model_dump()` to return the raw string value instead of the Enum object.
</details>

<details>
<summary><b>3. You configure a model with `use_enum_values=True` and define a field as `status: StatusEnum = StatusEnum.PENDING`. When you instantiate the model without arguments and call `model_dump()`, the field still shows up as the `StatusEnum.PENDING` object instead of the string `"pending"`. What caused this bug?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This is caused by Pydantic's default behavior of skipping validation on default values. <br><br>
The `use_enum_values` logic is applied during the validation phase. Because you instantiated the model using a default value, Pydantic skipped the validation phase entirely, bypassing the logic that extracts the string value from the Enum. To fix this, you must explicitly enable `validate_default=True` in your `model_config` so the default Enum object gets processed correctly.
</details>